# Lab 5: Binary Classification Problem (ML)

The goal of this lab is to train, evaluate, and optimize a binary classifier using Scikit-Learn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
from pydantic import BaseModel

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay

In [ ]:
# Define path for saving the model
MODEL_DIR = Path("../../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "titanic_rf_model.pkl"

print(f"Model will be saved to: {model_path.resolve()}")

## Loading and Initial Data Analysis

In this section, load the data from the `titanic.csv` file and explore its structure. Identify how many unique values are in each column, how the data is distributed, and what information we can extract from the dataset. Use the `read_csv` function to load the data.

In [ ]:
df = pd.read_csv('../data/titanic.csv')
df.head()

In [ ]:
# Display dataset information
df.info()

## Splitting Data into Training and Test Sets

In this section, split the data into test and training sets with an 80/20 ratio, keeping the following features: `'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'`. Use the `train_test_split` function for this:

In [ ]:
# Select features and target variable
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X = df[features]
y = df['Survived']

# Split data (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## Data Preprocessing

At this stage, we prepare a pipeline to transform input data for further analysis and model training. For this lab, a complete implementation is already provided.

In [ ]:
# Split features into numeric and categorical
numeric_features = ['Age', 'SibSp', 'Parch', 'Fare']
categorical_features = ['Pclass', 'Sex', 'Embarked']

# Pipeline for numeric features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline for categorical features
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', sparse_output=False))
])

# Combine pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Preprocessor ready!")

## Model Training

We'll start by selecting an algorithm for binary classification. You can choose from:
- Logistic regression
- Random forests
- Simple neural network (MLP)
- Support vector machine

In [ ]:
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.svm import SVC
# from sklearn.linear_model import LogisticRegression
# from sklearn.neural_network import MLPClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', model)
])

In [ ]:
# Train the model
pipeline.fit(X_train, y_train)
print("Model trained successfully!")

## Model Evaluation

At this stage, we'll calculate evaluation metrics for the trained model: `precision`, `recall`, `accuracy`, and `f1-score`. Assign model predictions for both datasets to variables `yhat_train` and `yhat_test`. Then, use the provided functions to calculate metrics and display the confusion matrix for the training set.

**NOTE**: In the markdown cell below the results, describe the obtained results, referring to the issues of high variance and bias (_overfitting_ and _underfitting_).

In [ ]:
class ClassificationMetrics(BaseModel):
    """Classification metrics"""
    accuracy: float
    precision: float
    recall: float
    f1: float
    
    def display(self, title: str = "Metrics"):
        """Display metrics in a readable format"""
        print(f"\n{'='*50}")
        print(f"{title}")
        print(f"{'='*50}")
        print(f"Accuracy:  {self.accuracy:.4f}")
        print(f"Precision: {self.precision:.4f}")
        print(f"Recall:    {self.recall:.4f}")
        print(f"F1-Score:  {self.f1:.4f}")


def calculate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> ClassificationMetrics:
    """Calculate classification metrics"""
    return ClassificationMetrics(
        accuracy=accuracy_score(y_true, y_pred),
        precision=precision_score(y_true, y_pred),
        recall=recall_score(y_true, y_pred),
        f1=f1_score(y_true, y_pred)
    )


def plot_confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, title: str = "Confusion Matrix"):
    """Draw confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Perished', 'Survived'])
    disp.plot(cmap='Blues')
    plt.title(title)
    plt.show()

In [ ]:
# Calculate model predictions for each set
yhat_train = pipeline.predict(X_train)
yhat_test = pipeline.predict(X_test)

# Calculate model metrics for each set
train_metrics = calculate_metrics(y_train, yhat_train)
test_metrics = calculate_metrics(y_test, yhat_test)

# Display metrics
train_metrics.display("Metrics - Training Set")
test_metrics.display("Metrics - Test Set")

In [ ]:
# Display confusion matrices
plot_confusion_matrix(y_train, yhat_train, "Confusion Matrix - Training Set")

In [ ]:
plot_confusion_matrix(y_test, yhat_test, "Confusion Matrix - Test Set")

## Model Optimization

In this section, we'll perform model optimization using `GridSearchCV` and `RandomSearchCV`. The goal is to identify the best classifier and compare its performance with the originally trained one. Our optimization metric is the `f1` score.

In [ ]:
# Parameters to search
param_distributions = {
    'classifier__n_estimators': [50, 100, 200, 300],
    'classifier__max_depth': [None, 10, 20, 30, 40],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__max_features': ['sqrt', 'log2', None]
}

random_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    scoring='f1'
)

print("Starting RandomizedSearchCV...")
random_search.fit(X_train, y_train)

print(f"\nBest parameters: {random_search.best_params_}")
print(f"Best CV score: {random_search.best_score_:.4f}")
print(f"Test set score: {random_search.score(X_test, y_test):.4f}")

In [ ]:
# Narrower parameter range (adjust based on RandomizedSearchCV results)
param_grid = {
    'classifier__n_estimators': [100, 150, 200],
    'classifier__max_depth': [10, 20, 30],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    verbose=1,
    scoring='f1'
)

print("Starting GridSearchCV...")
grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")
print(f"Test set score: {grid_search.score(X_test, y_test):.4f}")

In [ ]:
# Use the best model from GridSearchCV
best_model = grid_search.best_estimator_

# Predictions
yhat_test_optimized = best_model.predict(X_test)

# Metrics
optimized_metrics = calculate_metrics(y_test, yhat_test_optimized)
optimized_metrics.display("Metrics - Optimized Model")

# Confusion matrix
plot_confusion_matrix(y_test, yhat_test_optimized, "Confusion Matrix - Optimized Model")

In [ ]:
# Save model to file
joblib.dump(best_model, model_path)
print(f"Model saved at: {model_path.resolve()}")

## Using the Trained Model in Practice

In [ ]:
class Passenger(BaseModel):
    """Passenger data model"""
    Pclass: int
    Sex: str
    Age: float
    SibSp: int
    Parch: int
    Fare: float
    Embarked: str

def predict_survival(model_path: str | Path, passenger: Passenger) -> bool:
    """Load model and predict passenger survival"""
    model = joblib.load(model_path)
    passenger_df = pd.DataFrame([passenger.model_dump()])
    prediction = model.predict(passenger_df)[0]
    return bool(prediction)

print("Passenger model and prediction function ready!")

In [ ]:
# Example 1: Female, first class
passenger1 = Passenger(
    Pclass=1,
    Sex='female',
    Age=30,
    SibSp=0,
    Parch=0,
    Fare=100.0,
    Embarked='S'
)

result1 = predict_survival(model_path, passenger1)
print(f"Passenger 1 (female, 1st class): {'SURVIVES ✓' if result1 else 'PERISHES ✗'}")

In [ ]:
# Example 2: Male, third class
passenger2 = Passenger(
    Pclass=3,
    Sex='male',
    Age=25,
    SibSp=0,
    Parch=0,
    Fare=7.5,
    Embarked='S'
)

result2 = predict_survival(model_path, passenger2)
print(f"Passenger 2 (male, 3rd class): {'SURVIVES ✓' if result2 else 'PERISHES ✗'}")

In [ ]:
# Example 3: Child with family
passenger3 = Passenger(
    Pclass=2,
    Sex='female',
    Age=5,
    SibSp=1,
    Parch=2,
    Fare=30.0,
    Embarked='C'
)

result3 = predict_survival(model_path, passenger3)
print(f"Passenger 3 (child with family, 2nd class): {'SURVIVES ✓' if result3 else 'PERISHES ✗'}")